# Backpropagation and Optimization

Forward propagation computes predictions. **Backpropagation** computes how each weight and bias contributed to the error — the gradients needed to improve the network. **Optimization** uses those gradients to update parameters and reduce the loss.

Together, backpropagation and optimization form the training engine of every neural network. This notebook derives backpropagation step by step, implements it in NumPy, and covers the optimization algorithms — from basic gradient descent to Adam — that make deep learning practical.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. The Learning Problem

Training a neural network means finding weights and biases $\theta = \{\mathbf{W}^{[l]}, \mathbf{b}^{[l]}\}$ that minimize the loss:

$$\theta^* = \arg\min_\theta \mathcal{L}(\theta) = \arg\min_\theta \frac{1}{m} \sum_{i=1}^{m} L(f_\theta(x_i), y_i)$$

The network defines a function $f_\theta$. The loss measures error. Learning adjusts $\theta$ to reduce error.

**Gradient descent** is the core algorithm:

$$\theta \leftarrow \theta - \eta \frac{\partial \mathcal{L}}{\partial \theta}$$

where $\eta$ is the **learning rate**. The challenge: a network may have millions of parameters. Computing $\frac{\partial \mathcal{L}}{\partial \theta}$ for each one efficiently is what backpropagation solves.

---

## 2. The Chain Rule

A neural network is a composition of functions. The chain rule from calculus provides the key:

If $y = f(g(x))$, then:

$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$$

For a network with layers $x \to z^{[1]} \to a^{[1]} \to z^{[2]} \to a^{[2]} \to \hat{y} \to L$:

$$\frac{\partial L}{\partial W^{[1]}} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial a^{[2]}} \cdot \frac{\partial a^{[2]}}{\partial z^{[2]}} \cdot \frac{\partial z^{[2]}}{\partial a^{[1]}} \cdot \frac{\partial a^{[1]}}{\partial z^{[1]}} \cdot \frac{\partial z^{[1]}}{\partial W^{[1]}}$$

Backpropagation applies the chain rule systematically from the output backward to the input, reusing intermediate results to avoid redundant computation.

---

## 3. Backpropagation Intuition

Think of the network as a pipeline. Forward pass: data flows input → output, computing predictions. Backward pass: error flows output → input, assigning blame to each parameter.

Define $\delta^{[l]} = \frac{\partial \mathcal{L}}{\partial z^{[l]}}$ — the gradient of the loss with respect to the pre-activation of layer $l$. This measures how much a small change in $z^{[l]}$ affects the loss.

**Backward pass steps:**

1. **Output layer:** Compute $\delta^{[L]}$ from the loss and output activation.
2. **Hidden layers:** Propagate $\delta^{[l+1]}$ backward through weights and activations to get $\delta^{[l]}$.
3. **Gradients:** Use $\delta^{[l]}$ to compute $\frac{\partial \mathcal{L}}{\partial W^{[l]}}$ and $\frac{\partial \mathcal{L}}{\partial b^{[l]}}$.

```
Forward:   x → z¹ → a¹ → z² → a² → ŷ → L
Backward:  x ← δ¹ ← a¹ ← δ² ← a² ← δ³ ← ŷ ← L
```

---

## 4. Backpropagation Equations

For a network with sigmoid output and binary cross-entropy loss (the combined gradient simplifies):

**Output layer:**

$$\delta^{[L]} = \mathbf{a}^{[L]} - \mathbf{y}$$

**Hidden layers** (for sigmoid/tanh activations with derivative $f'(z)$):

$$\delta^{[l]} = \left((\mathbf{W}^{[l+1]})^T \delta^{[l+1]}\right) \odot f'(z^{[l]})$$

where $\odot$ is element-wise multiplication.

For **ReLU**: $f'(z) = 1$ if $z > 0$, else $0$.

**Parameter gradients:**

$$\frac{\partial \mathcal{L}}{\partial W^{[l]}} = \frac{1}{m} \delta^{[l]} (\mathbf{a}^{[l-1]})^T$$

$$\frac{\partial \mathcal{L}}{\partial b^{[l]}} = \frac{1}{m} \sum_{i=1}^{m} \delta^{[l]}_i$$

These four equations — output delta, hidden delta, weight gradient, bias gradient — are the complete backpropagation algorithm.

---

## 5. Implementing Backpropagation

We implement a complete feedforward network with forward pass, backward pass, and training loop.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)


class NeuralNet:
    """2-layer network with backpropagation."""

    def __init__(self, n_input, n_hidden, n_output):
        self.W1 = np.random.randn(n_hidden, n_input) * np.sqrt(2.0 / n_input)
        self.b1 = np.zeros((n_hidden, 1))
        self.W2 = np.random.randn(n_output, n_hidden) * np.sqrt(2.0 / n_hidden)
        self.b2 = np.zeros((n_output, 1))

    def forward(self, X):
        self.z1 = self.W1 @ X + self.b1
        self.a1 = relu(self.z1)
        self.z2 = self.W2 @ self.a1 + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    def backward(self, X, Y):
        m = X.shape[1]
        dz2 = self.a2 - Y                          # sigmoid + BCE combined
        dW2 = (1/m) * dz2 @ self.a1.T
        db2 = (1/m) * np.sum(dz2, axis=1, keepdims=True)
        dz1 = (self.W2.T @ dz2) * relu_grad(self.z1)
        dW1 = (1/m) * dz1 @ X.T
        db1 = (1/m) * np.sum(dz1, axis=1, keepdims=True)
        return dW1, db1, dW2, db2

    def update(self, grads, lr):
        dW1, db1, dW2, db2 = grads
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2

    def loss(self, Y, Y_hat):
        m = Y.shape[1]
        eps = 1e-15
        Y_hat = np.clip(Y_hat, eps, 1 - eps)
        return -np.sum(Y * np.log(Y_hat) + (1 - Y) * np.log(1 - Y_hat)) / m

    def predict(self, X):
        return (self.forward(X) > 0.5).astype(int)

In [ ]:
# Train on moons dataset
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_tr_t = scaler.fit_transform(X_tr).T   # (2, m)
X_te_t = scaler.transform(X_te).T
Y_tr = y_tr.reshape(1, -1)

nn = NeuralNet(2, 16, 1)
losses, train_accs = [], []

for epoch in range(2000):
    Y_hat = nn.forward(X_tr_t)
    loss = nn.loss(Y_tr, Y_hat)
    grads = nn.backward(X_tr_t, Y_tr)
    nn.update(grads, lr=0.5)
    losses.append(loss)
    if epoch % 200 == 0:
        acc = accuracy_score(y_tr, nn.predict(X_tr_t).flatten())
        train_accs.append(acc)

test_acc = accuracy_score(y_te, nn.predict(X_te_t).flatten())
print(f"Final loss: {losses[-1]:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses, linewidth=1)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss decreases")

plt.subplot(1, 2, 2)
xx, yy = np.meshgrid(np.linspace(-2, 3, 150), np.linspace(-1.5, 2, 150))
grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()]).T
Z = nn.forward(grid)
plt.contourf(xx, yy, Z.reshape(xx.shape), alpha=0.3, cmap="coolwarm")
plt.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", edgecolors="k", s=15)
plt.title(f"Decision boundary (acc={test_acc:.3f})")
plt.tight_layout()
plt.show()

---

## 6. Gradient Descent Variants

How we use gradients to update weights defines the optimization strategy.

### 6.1 Batch Gradient Descent

Compute gradients on the **entire** training set, then update:

$$\theta \leftarrow \theta - \eta \frac{1}{m} \sum_{i=1}^{m} \nabla_\theta L_i$$

Stable convergence but slow for large datasets. Each update requires a full pass through all data.

### 6.2 Stochastic Gradient Descent (SGD)

Update after **each individual example**:

$$\theta \leftarrow \theta - \eta \nabla_\theta L_i$$

Fast updates, noisy gradients. Can escape local minima due to noise. Convergence is erratic.

### 6.3 Mini-Batch Gradient Descent

The practical standard: update after a **mini-batch** of $B$ examples (typically 32, 64, 128, 256):

$$\theta \leftarrow \theta - \eta \frac{1}{B} \sum_{i \in \text{batch}} \nabla_\theta L_i$$

Balances stability and speed. Mini-batches fit in GPU memory and exploit parallel computation. Our implementation above uses full-batch (all training data) for simplicity.

---

## 7. The Learning Rate

The learning rate $\eta$ controls the step size. It is the most important hyperparameter.

- **Too large** — overshoots the minimum, loss oscillates or diverges.
- **Too small** — slow convergence, may get stuck in poor local minima.
- **Just right** — steady decrease in loss, efficient convergence.

There is no universal optimal value. It depends on the model, data, optimizer, and batch size.

In [ ]:
# Learning rate effect on convergence
def train_with_lr(lr, epochs=500):
    net = NeuralNet(2, 8, 1)
    loss_history = []
    for _ in range(epochs):
        Y_hat = net.forward(X_tr_t)
        loss_history.append(net.loss(Y_tr, Y_hat))
        net.update(net.backward(X_tr_t, Y_tr), lr)
    return loss_history

plt.figure(figsize=(9, 4))
for lr, color in [(0.01, "green"), (0.5, "blue"), (5.0, "red")]:
    plt.plot(train_with_lr(lr), color=color, label=f"lr={lr}", linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning rate: too small (slow), good, too large (diverges)")
plt.legend()
plt.show()

---

## 8. Momentum

SGD can oscillate in narrow valleys and progress slowly along flat directions. **Momentum** accumulates a velocity vector:

$$v_t = \beta v_{t-1} + (1 - \beta) \nabla_\theta \mathcal{L}$$

$$\theta \leftarrow \theta - \eta v_t$$

where $\beta$ (typically 0.9) is the momentum coefficient. The update accumulates past gradients — like a ball rolling downhill, gaining speed in consistent directions and dampening oscillations in others.

**Nesterov momentum** — look ahead before computing the gradient, correcting for overshooting. Often converges faster than standard momentum.

In [ ]:
# Simple 2D optimization: SGD vs Momentum
def rosenbrock(xy):
    x, y = xy
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(xy):
    x, y = xy
    dx = -2*(1-x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

def optimize(method, lr=0.001, beta=0.9, steps=2000):
    pos = np.array([-1.0, 1.0])
    path = [pos.copy()]
    v = np.zeros(2)
    for _ in range(steps):
        grad = rosenbrock_grad(pos)
        if method == "sgd":
            pos -= lr * grad
        else:
            v = beta * v + lr * grad
            pos -= v
        path.append(pos.copy())
    return np.array(path)

path_sgd = optimize("sgd")
path_mom = optimize("momentum")

plt.figure(figsize=(8, 6))
plt.plot(path_sgd[:, 0], path_sgd[:, 1], "b-", alpha=0.5, label="SGD", linewidth=1)
plt.plot(path_mom[:, 0], path_mom[:, 1], "r-", alpha=0.7, label="Momentum", linewidth=1.5)
plt.plot(1, 1, "g*", markersize=15, label="Minimum")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Momentum navigates valleys faster than SGD")
plt.legend()
plt.show()

---

## 9. Adam Optimizer

**Adam** (Adaptive Moment Estimation) combines momentum with per-parameter adaptive learning rates. It is the default optimizer for most deep learning work.

Adam maintains two moving averages for each parameter:

- $m_t$ — first moment (mean of gradients) — like momentum.
- $v_t$ — second moment (uncentered variance of gradients) — adapts learning rate per parameter.

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$

$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

Bias correction (important in early steps):

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

Update:

$$\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Default hyperparameters: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, $\eta = 0.001$.

Adam adapts: parameters with large gradients get smaller effective learning rates; parameters with small gradients get larger ones. This handles sparse gradients and different parameter scales well.

In [ ]:
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1, self.beta2, self.eps = beta1, beta2, eps
        self.t = 0
        self.m, self.v = {}, {}

    def step(self, params, grads):
        self.t += 1
        updated = {}
        for key in params:
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * grads[key]**2
            m_hat = self.m[key] / (1 - self.beta1**self.t)
            v_hat = self.v[key] / (1 - self.beta2**self.t)
            updated[key] = params[key] - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return updated


# Train with Adam
net = NeuralNet(2, 16, 1)
optimizer = Adam(lr=0.01)
adam_losses = []

for epoch in range(2000):
    Y_hat = net.forward(X_tr_t)
    adam_losses.append(net.loss(Y_tr, Y_hat))
    dW1, db1, dW2, db2 = net.backward(X_tr_t, Y_tr)
    params = {"W1": net.W1, "b1": net.b1, "W2": net.W2, "b2": net.b2}
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    updated = optimizer.step(params, grads)
    net.W1, net.b1, net.W2, net.b2 = updated["W1"], updated["b1"], updated["W2"], updated["b2"]

print(f"Adam final loss: {adam_losses[-1]:.4f}")
print(f"Adam test accuracy: {accuracy_score(y_te, net.predict(X_te_t).flatten()):.4f}")

---

## 10. Other Optimizers

| Optimizer | Key Idea |
|-----------|----------|
| **SGD** | Basic gradient descent |
| **SGD + Momentum** | Accumulated velocity |
| **RMSprop** | Adapts learning rate using recent gradient magnitudes |
| **Adam** | Momentum + RMSprop combined |
| **AdamW** | Adam with decoupled weight decay (better regularization) |
| **Lion** | Sign-based updates, memory efficient |

In practice, **Adam** or **AdamW** is the default starting point. SGD with momentum sometimes achieves better final performance with careful tuning but requires more effort.

---

## 11. Learning Rate Schedules

A fixed learning rate may be too large at the end of training (oscillating around the minimum) or too small at the beginning (slow progress). **Learning rate schedules** adjust $\eta$ during training.

**Step decay** — reduce $\eta$ by a factor every $N$ epochs. Simple and effective.

**Exponential decay** — $\eta_t = \eta_0 \cdot e^{-\lambda t}$. Smooth reduction.

**Cosine annealing** — $\eta$ follows a cosine curve from initial value to near zero. Popular in modern training.

**Warmup** — start with a small $\eta$ and linearly increase for the first few epochs. Stabilizes early training for large models and batch sizes.

**Reduce on plateau** — decrease $\eta$ when validation loss stops improving.

In [ ]:
# Learning rate schedules
epochs = np.arange(100)
lr_init = 0.1

step = lr_init * 0.5**(epochs // 30)
cosine = lr_init * 0.5 * (1 + np.cos(np.pi * epochs / 100))
warmup = np.where(epochs < 10, lr_init * epochs / 10, lr_init)

plt.figure(figsize=(9, 4))
plt.plot(epochs, step, label="Step decay", linewidth=2)
plt.plot(epochs, cosine, label="Cosine annealing", linewidth=2)
plt.plot(epochs, warmup, label="Warmup (first 10)", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("Common learning rate schedules")
plt.legend()
plt.show()

---

## 12. Vanishing and Exploding Gradients

During backpropagation, gradients are multiplied through each layer. This can cause two problems:

**Vanishing gradients** — gradients shrink toward zero in early layers. Early layers learn extremely slowly or not at all. Caused by sigmoid/tanh activations (derivative ≤ 0.25) and deep networks.

**Exploding gradients** — gradients grow exponentially. Weight updates become enormous, causing NaN values and training failure. Common in RNNs processing long sequences.

**Solutions:**
- Use **ReLU** activations (gradient = 1 for positive inputs).
- **Residual connections** (ResNet) — skip connections let gradients flow directly.
- **Batch normalization** — stabilizes layer inputs.
- **Gradient clipping** — cap gradient magnitude: if $\|g\| > \text{threshold}$, scale down.
- **Careful initialization** (He, Xavier).
- **LSTM/GRU** gates for sequences.

In [ ]:
# Gradient magnitude through layers: ReLU vs Sigmoid
n_layers = 15
grad_relu, grad_sig = 1.0, 1.0
relu_history, sig_history = [1.0], [1.0]

for _ in range(n_layers):
    grad_relu *= 1.0    # ReLU derivative ≈ 1 (when active)
    grad_sig *= 0.25    # sigmoid derivative max = 0.25
    relu_history.append(grad_relu)
    sig_history.append(grad_sig)

plt.figure(figsize=(8, 4))
plt.plot(relu_history, "r-o", label="ReLU (grad ≈ 1)", markersize=4)
plt.plot(sig_history, "b-o", label="Sigmoid (grad ≤ 0.25)", markersize=4)
plt.xlabel("Layers (backward)")
plt.ylabel("Gradient magnitude")
plt.title("Sigmoid causes vanishing gradients in deep networks")
plt.yscale("log")
plt.legend()
plt.show()

---

## 13. The Complete Training Loop

Putting it all together — the training loop used in every deep learning framework:

In [ ]:
def train_network(X, Y, n_hidden=16, epochs=3000, lr=0.5, print_every=500):
    """Complete training loop with backprop and gradient descent."""
    net = NeuralNet(X.shape[0], n_hidden, Y.shape[0])
    history = {"loss": [], "accuracy": []}

    for epoch in range(1, epochs + 1):
        # Forward pass
        Y_hat = net.forward(X)

        # Compute loss
        loss = net.loss(Y, Y_hat)

        # Backward pass
        grads = net.backward(X, Y)

        # Update weights
        net.update(grads, lr)

        history["loss"].append(loss)

        if epoch % print_every == 0:
            acc = np.mean(net.predict(X) == Y)
            history["accuracy"].append(acc)
            print(f"Epoch {epoch:5d} | Loss: {loss:.4f} | Accuracy: {acc:.4f}")

    return net, history

net, history = train_network(X_tr_t, Y_tr)
print(f"\nTest accuracy: {accuracy_score(y_te, net.predict(X_te_t).flatten()):.4f}")

---

## 14. Summary

**Backpropagation** applies the chain rule to compute gradients of the loss with respect to every weight and bias. It works backward from the output layer, propagating error signals through each layer using the $\delta$ values. The key equations: output delta, hidden delta via weight transpose and activation derivative, and parameter gradients from delta and activations.

**Optimization** uses gradients to update parameters. Gradient descent variants (batch, stochastic, mini-batch) trade off stability and speed. **Momentum** accelerates convergence in consistent directions. **Adam** adapts learning rates per parameter and is the default choice. **Learning rate schedules** fine-tune training dynamics over time.

The training loop — forward, loss, backward, update — repeated over epochs, is how every neural network learns. Vanishing and exploding gradients remain challenges addressed by activation choice, architecture design, and gradient clipping.